In [1]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine

### 1. SQL Server

In [6]:
server = "127.0.0.1,1500"
# database = "DEP2_staging"
database = "DEP2"
username = "sa"
password = "dep2025-G12"
driver = "ODBC Driver 17 for SQL Server"

conn = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)

conn = pyodbc.connect(conn)

### 2. Load FactWifiConnection

In [3]:
wifi_query = """
SELECT f.DateKey, f.TimeKey, f.UserKey, b.SubgroupKey
FROM dbo.FactWifiConnection f
JOIN dbo.BridgeUserSubgroup b
    ON f.UserKey = b.UserKey
"""
wifi_df = pd.read_sql(wifi_query, conn)

C:\Users\Semih\AppData\Local\Temp\ipykernel_23336\2191648252.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  wifi_df = pd.read_sql(wifi_query, conn)


In [4]:
print(wifi_df.head())

    DateKey  TimeKey  UserKey  SubgroupKey
0  20251015   133400       29      5718103
1  20251015   133400       29      5718102
2  20251015   133400       29      5718101
3  20251015   133400       29      5718097
4  20251015   133400       29      5717964


### 3. Aggregate attendance

In [5]:
attendance_df = (
    wifi_df.groupby(['DateKey', 'TimeKey', 'SubgroupKey'])
    .agg(PresentStudents=('UserKey', 'nunique'))
    .reset_index()
)

# attendance_df = attendance_df[attendance_df['SubgroupKey'] == 5816398]
attendance_df

,DateKey,TimeKey,SubgroupKey,PresentStudents
0,20251015,133400,5717941,1
1,20251015,133400,5717964,1
2,20251015,133400,5718097,1
3,20251015,133400,5718101,1
4,20251015,133400,5718102,1
...,...,...,...,...
534713,20251110,155600,5976932,1
534714,20251110,155600,5983541,1
534715,20251110,155600,6005089,1
534716,20251110,155600,6005091,2


In [ ]:
# # Subgroup linken met SubgroupKey
# subgroup_query = """
# SELECT SubgroupKey, SubgroupCode, SubgroupName
# FROM DEP2.dbo.DimSubgroup
# """
# dim_subgroup_df = pd.read_sql(subgroup_query, conn)
# dim_subgroup_df

# Subgroup linken met SubgroupKey
subgroup_query = """
SELECT SubgroupKey, SubgroupCode
FROM DEP2.dbo.DimSubgroup
"""
dim_subgroup_df = pd.read_sql(subgroup_query, conn)
dim_subgroup_df

C:\Users\Semih\AppData\Local\Temp\ipykernel_23336\116916421.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_subgroup_df = pd.read_sql(subgroup_query, conn)


,SubgroupKey,SubgroupCode
0,5717692,PBA-SO/LOBR/2A
1,5717824,PBA-SO/LOBR/2A
2,5717825,PBA-SO/LOBR/2A
3,5717826,PBA-SO/LOBR/2A
4,5717829,PBA-SO/LOBR/2A
...,...,...
6902,6160650,GRAD-LABT/DAG/1D
6903,6160651,GRAD-LABT/DAG/1D
6904,6160654,GRAD-LABT/DAG/1D
6905,6160659,GRAD-LABT/DAG/1D


In [ ]:
# attendance_df = attendance_df.merge(
#     dim_subgroup_df[['SubgroupCode', 'SubgroupName']],
#     left_on='SubgroupKey',   
#     right_on='SubgroupCode', 
#     how='left'
# )

attendance_df = attendance_df.merge(
    dim_subgroup_df,
    on='SubgroupKey',
    how='left'
)

# attendance_df = attendance_df[['DateKey', 'TimeKey', 'SubgroupKey', 'SubgroupName', 'PresentStudents']]
attendance_df = attendance_df[['DateKey', 'TimeKey', 'SubgroupKey', 'SubgroupCode', 'PresentStudents']]

In [12]:
attendance_df

,DateKey,TimeKey,SubgroupKey,SubgroupCode,PresentStudents
0,20251015,133400,5717941,PBA-SO/LOBR/2B,1
1,20251015,133400,5717964,PBA-SO/LOBR/2B,1
2,20251015,133400,5718097,PBA-SO/LOBR/2B,1
3,20251015,133400,5718101,PBA-SO/LOBR/2B,1
4,20251015,133400,5718102,PBA-SO/LOBR/2B,1
...,...,...,...,...,...
534713,20251110,155600,5976932,PBA-EM-EM/3D1,1
534714,20251110,155600,5983541,PBA-EM-EM/2E2,1
534715,20251110,155600,6005089,PBA-SW/AO/2,1
534716,20251110,155600,6005091,PBA-SW/AO/2,2
